In [1]:
!pip install --ignore-requires-python syncode==0.4.16

In [2]:
from syncode import Syncode

## Grammar

In [3]:
grammar = r"""
start: " "? "Operands: " number ", " number "; Operator: " operator
number: /-?[0-9]+(\.[0-9]*)?/
operator: "+" | "-" | "*" | "/"
"""

## Few-shot prompt

In [4]:
few_shot = """Extract the two operands and the operator from each math question.

Question: What is 25. multiplied by 4.0?
Answer: Operands: 25.0, 4.0; Operator: *

Question: What is 13.7 plus 6.25?
Answer: Operands: 13.7, 6.25; Operator: +

Question: What is 81.0 divided by 9.0?
Answer: Operands: 81.0, 9.0; Operator: /

Question: What is 50.5 minus 20.75?
Answer: Operands: 50.5, 20.75; Operator: -

"""

## Load Phi-2 with SynCode

In [5]:
syn_llm = Syncode(model="microsoft/phi-2", grammar=grammar)

[2026-10-27 20:48:54,144-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[2026-10-27 20:48:57,859-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## run_syncode

In [6]:
def run_syncode(textual_query):
    prompt = few_shot + "Question: " + textual_query + "\nAnswer:"
    output = syn_llm.infer(prompt)[0]

    operands_part, operator_part = output.split(";")
    numbers = operands_part.replace("Operands:", "").split(",")
    a = float(numbers[0])
    b = float(numbers[1])
    operator = operator_part.replace("Operator:", "").strip()

    if operator == "+":
        answer = a + b
    elif operator == "-":
        answer = a - b
    elif operator == "*":
        answer = a * b
    else:
        answer = a / b

    return ([a, b], operator, float(answer))

## Test

In [7]:
print(run_syncode("What is 327. multiplied by 11.0?"))
print(run_syncode("What is 45.1 plus 23.54?"))
print(run_syncode("What is 120.4 divided by 4.0?"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


([327.0, 11.0], '*', 3597.0)
([45.1, 23.54], '+', 68.64)
([120.4, 4.0], '/', 30.1)
